# 🛡️ Autonomous SecOps AI Agent - Phase 1: Data Ingestion

> **Dataset:** RT-IoT2022 (UCI Machine Learning Repository ID: 942)  
> **Objective:** Load IoT network telemetry flow records, verify schema and dimensions, inspect missing values, and validate target class counts.

## 1. Environment & Library Imports

In [1]:
import os
import sys
import pandas as pd
import numpy as np

# Display options
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 50)
print(f"Pandas Version: {pd.__version__}")
print(f"NumPy Version:  {np.__version__}")

Pandas Version: 3.1.1
NumPy Version:  2.5.5


## 2. Locate and Load Dataset
We check the local `data/` directory first for instant loading. If missing, we automatically fetch it via `ucimlrepo`.

In [2]:
def get_dataset_path():
    candidate_paths = [
        "../data/RT_IOT2022.csv",
        "data/RT_IOT2022.csv",
        os.path.abspath(os.path.join(os.getcwd(), "..", "data", "RT_IOT2022.csv")),
    ]
    for path in candidate_paths:
        if os.path.exists(path):
            return path
    
    # Fallback to ucimlrepo fetch
    print("Local file not found, fetching via ucimlrepo (ID: 942)...")
    from ucimlrepo import fetch_ucirepo
    repo = fetch_ucirepo(id=942)
    df_remote = pd.concat([repo.data.features, repo.data.targets], axis=1)
    os.makedirs("../data", exist_ok=True)
    save_path = "../data/RT_IOT2022.csv"
    df_remote.to_csv(save_path, index=False)
    return save_path

data_path = get_dataset_path()
print(f"Loading dataset from: {data_path} ...")
df = pd.read_csv(data_path)

# Drop redundant raw index column if present
if "Unnamed: 0" in df.columns:
    df.drop(columns=["Unnamed: 0"], inplace=True)

print(f"✓ Dataset successfully loaded! Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")

Loading dataset from: data/RT_IOT2022.csv ...
✓ Dataset successfully loaded! Shape: 123,117 rows × 84 columns


## 3. Dataset Dimensions & Preview
Inspect the first few network flow telemetry records.

In [3]:
print(f"Total Network Flow Records: {df.shape[0]:,}")
print(f"Total Features (Excluding Target): {df.shape[1] - 1}")
df.head()

Total Network Flow Records: 123,117
Total Features (Excluding Target): 83


## 4. Data Quality & Missing Value Inspection

In [4]:
null_counts = df.isnull().sum()
total_nulls = null_counts.sum()
print(f"Total Missing / NaN Values in Entire Dataset: {total_nulls}")
if total_nulls > 0:
    print("Columns with missing values:")
    print(null_counts[null_counts > 0])
else:
    print("✓ The dataset has zero missing values. Highly complete!")

# Data types summary
print("\nFeature Data Types:")
print(df.dtypes.value_counts())

Total Missing / NaN Values in Entire Dataset: 0
✓ The dataset has zero missing values. Highly complete!

Feature Data Types:
float64    56
int64      25
str         3
Name: count, dtype: int64


## 5. Target Class Distribution Analysis
The dataset contains 12 verified classes covering DoS/DDoS attacks, reconnaissance/port scans, MITM/brute force, and benign smart-home IoT traffic.

In [5]:
# Compute counts and percentages
counts = df["Attack_type"].value_counts()
percentages = (df["Attack_type"].value_counts(normalize=True) * 100).round(3)

# Categorize broad threat types
def categorize_threat(name):
    if name in ["DOS_SYN_Hping", "DDOS_Slowloris"]:
        return "DoS / DDoS Attack"
    elif name.startswith("NMAP_"):
        return "Reconnaissance / Scan"
    elif name in ["ARP_poisioning", "Metasploit_Brute_Force_SSH"]:
        return "MITM / Brute Force"
    elif name in ["Thing_Speak", "MQTT_Publish", "Wipro_bulb"]:
        return "Benign IoT Traffic"
    return "Other"

summary_df = pd.DataFrame({
    "Count": counts,
    "Percentage (%)": percentages,
    "Threat Category": [categorize_threat(idx) for idx in counts.index]
})

print(summary_df)


                            Count  Percentage (%)        Threat Category
Attack_type                                                             
DOS_SYN_Hping               94659          76.885      DoS / DDoS Attack
Thing_Speak                  8108           6.586     Benign IoT Traffic
ARP_poisioning               7750           6.295     MITM / Brute Force
MQTT_Publish                 4146           3.368     Benign IoT Traffic
NMAP_UDP_SCAN                2590           2.104  Reconnaissance / Scan
NMAP_XMAS_TREE_SCAN          2010           1.633  Reconnaissance / Scan
NMAP_OS_DETECTION            2000           1.624  Reconnaissance / Scan
NMAP_TCP_scan                1002           0.814  Reconnaissance / Scan
DDOS_Slowloris                534           0.434      DoS / DDoS Attack
Wipro_bulb                    253           0.205     Benign IoT Traffic
Metasploit_Brute_Force_SSH     37           0.030     MITM / Brute Force
NMAP_FIN_SCAN                  28           0.023  

## 6. Categorical vs. Numerical Telemetry Breakdown

In [6]:
cat_cols = df.select_dtypes(include=["object"]).columns.tolist()
num_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()

print(f"Categorical Columns ({len(cat_cols)}): {cat_cols}")
for col in [c for c in cat_cols if c != "Attack_type"]:
    print(f"Unique values in {col}: {df[col].unique().tolist()}")

print(f"\nNumerical Telemetry Features: {len(num_cols)} features")
print("Sample Numerical Features:", num_cols[:10])

Categorical Columns (3): ['proto', 'service', 'Attack_type']
Unique values in proto: ['tcp', 'udp', 'icmp']
Unique values in service: ['mqtt', '-', 'http', 'dns', 'ntp', 'ssl', 'dhcp', 'irc', 'ssh', 'radius']

Numerical Telemetry Features: 81 features
Sample Numerical Features: ['id.orig_p', 'id.resp_p', 'flow_duration', 'fwd_pkts_tot', 'bwd_pkts_tot', 'fwd_data_pkts_tot', 'bwd_data_pkts_tot', 'fwd_pkts_per_sec', 'bwd_pkts_per_sec', 'flow_pkts_per_sec']


## 7. Key Findings & Next Steps
1. **Volume & Integrity:** 123,117 network flow records with 0 missing values.
2. **Class Imbalance:** `DOS_SYN_Hping` represents ~76.9% of traffic, while minority attacks like `Metasploit_Brute_Force_SSH` (37 records) and `NMAP_FIN_SCAN` (28 records) are rare.
3. **Next Step:** Feature encoding, scaling, train/test split, and multi-class intrusion classification model training.